# Day 11 - Reading CSV files with the csv module

Tasks 3 to 8 and 10 from the assignment.

## Parameters
Change `INPUT_DIR` here instead of editing the code below.

In [ ]:
INPUT_DIR = "olist_data"  # folder that contains the Olist CSV files


## Helper functions\nShared by the tasks below. They accept comma- or semicolon-delimited files.

In [ ]:
import csv
import os

def detect_delimiter(path):
    """Return ',' or ';' depending on which one the file uses."""
    with open(path, newline="", encoding="utf-8") as f:
        sample = f.read(4096)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;").delimiter
    except csv.Error:
        return ","

def read_rows(path):
    """Return (header, rows) for a comma- or semicolon-delimited CSV."""
    delim = detect_delimiter(path)
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=delim)
        header = next(reader)
        return header, list(reader)


## Task 4 - Row count of olist_orders_dataset.csv

In [ ]:
ORDERS_FILE = os.path.join(INPUT_DIR, "olist_orders_dataset.csv")

header, rows = read_rows(ORDERS_FILE)
row_count = len(rows)

print("Row count (excluding header):", row_count)


## Task 5 - Column names and first 5 rows

In [ ]:
header, rows = read_rows(ORDERS_FILE)

print("Columns:", header)
print()
for row in rows[:5]:
    print(row)


## Task 6 - Missing values per column
A value counts as missing if it is empty or only whitespace. Short rows count as missing for the columns they don't reach.

In [ ]:
def count_missing(path):
    header, rows = read_rows(path)
    missing = {col: 0 for col in header}
    for row in rows:
        for i, col in enumerate(header):
            if i >= len(row) or row[i].strip() == "":
                missing[col] += 1
    return missing

for col, n in count_missing(ORDERS_FILE).items():
    print(f"{col}: {n}")


## Task 7 - Summary for every CSV file in the input folder

In [ ]:
def summarize_csv(path):
    header, rows = read_rows(path)
    missing_total = sum(
        1 for row in rows for i in range(len(header))
        if i >= len(row) or row[i].strip() == ""
    )
    return {"file": os.path.basename(path), "rows": len(rows),
            "columns": len(header), "missing_values": missing_total}

csv_files = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".csv"))
if not csv_files:
    print(f"No CSV files found in '{INPUT_DIR}'")

for name in csv_files:
    s = summarize_csv(os.path.join(INPUT_DIR, name))
    print(f"{s['file']}: {s['rows']} rows, {s['columns']} columns, {s['missing_values']} missing values")


## Task 10 - Copy a file with a different delimiter and read both formats
The copy uses `;` as the delimiter. The reader detects whether a file uses `,` or `;`, so both formats work.

In [ ]:
def copy_with_delimiter(src, dst, out_delim=";"):
    header, rows = read_rows(src)
    with open(dst, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=out_delim)
        writer.writerow(header)
        writer.writerows(rows)
    return dst

if csv_files:
    src = os.path.join(INPUT_DIR, csv_files[0])
    dst = os.path.join(INPUT_DIR, "copy_" + csv_files[0])
    copy_with_delimiter(src, dst, out_delim=";")
    print("Written:", dst)
    print("Detected delimiter for original:", repr(detect_delimiter(src)))
    print("Detected delimiter for copy:", repr(detect_delimiter(dst)))
